In [3]:
import weaviate
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
import pandas as pd
from datetime import datetime


c:\Users\d.dipadova\Documents\Uni\Magistrale\Corsi\2025-Marzo\BigData\Progetto_individuale\neuralTabb\neuralTab\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
### --- CONFIGURAZIONE --- ###
with open("config/configLLM.txt", "r") as f:
    GEMINI_KEY = f.read().strip()

genai.configure(api_key=GEMINI_KEY)


weaviate_client = weaviate.connect_to_local()
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")



['_WeaviateClientExecutor__close_async', '_WeaviateClientExecutor__parse_connection_params_and_embedded_db', '__annotations__', '__class__', '__class_getitem__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_connection', '_connection_type', '_is_protocol', 'backup', 'batch', 'close', 'cluster', 'collections', 'connect', 'debug', 'get_meta', 'get_open_id_configuration', 'graphql_raw_query', 'integrations', 'is_connected', 'is_live', 'is_ready', 'roles', 'users']


In [9]:
# Reset schema
import weaviate

weaviate_client = weaviate.connect_to_local()
if weaviate_client.collections.exists("BookData"):
    weaviate_client.collections.delete("BookData")
weaviate_client.close()

In [13]:
weaviate_client.close()

In [ ]:

from weaviate.classes.config import Configure
# vectorizer  = Dici a Weaviate di non generare automaticamente gli embeddings, li calcoliamo noi

weaviate_client.collections.create(

    name="BookData",
    properties=[
        wc.Property(name="film_code", data_type=wc.DataType.NUMBER),
        wc.Property(name="cinema_code", data_type=wc.DataType.NUMBER),
        wc.Property(name="total_sales", data_type=wc.DataType.NUMBER),
        wc.Property(name="tickets_sold", data_type=wc.DataType.NUMBER),
        wc.Property(name="tickets_out", data_type=wc.DataType.NUMBER),
        wc.Property(name="show_time", data_type=wc.DataType.NUMBER),
        wc.Property(name="occu_perc", data_type=wc.DataType.NUMBER),
        wc.Property(name="ticket_price", data_type=wc.DataType.NUMBER),
        wc.Property(name="ticket_use", data_type=wc.DataType.NUMBER),
        wc.Property(name="capacity", data_type=wc.DataType.NUMBER),
        wc.Property(name="date", data_type=wc.DataType.DATE),
        wc.Property(name="month", data_type=wc.DataType.NUMBER),
        wc.Property(name="quarter", data_type=wc.DataType.NUMBER),
        wc.Property(name="day", data_type=wc.DataType.NUMBER),
        wc.Property(name="raw_text", data_type=wc.DataType.TEXT),
    ],
    vectorizer_config=Configure.Vectorizer.text2vec_transformers()
)


NameError: name 'weaviate_client' is not defined

In [12]:
# Get a collection
collection = weaviate_client.collections.get("MovieStats")

collection.exists()  # Check if the collection exists


False

In [23]:

# Reset schema
import weaviate

weaviate_client = weaviate.connect_to_local()






In [26]:
    # Basic search (fetch objects)
collection = weaviate_client.collections.get("book_data")
response = collection.query.fetch_objects(
        limit=100,
        return_properties=["title", "authors", "description"]
    )

for item in response.objects:
    print(item.properties)  # Print the raw text of each item

weaviate_client.close()



In [ ]:


def row_to_text(row):
    return (
        f"Il {row['date']} il film {row['film_code']} è stato proiettato al cinema {row['cinema_code']} "
        f"alle {row['show_time']}. Sono stati venduti {row['tickets_sold']} biglietti su una capacità di "
        f"{row['capacity']} posti (occupazione: {row['occu_perc']}%). "
        f"Incasso totale: {row['total_sales']} euro, prezzo medio del biglietto: {row['ticket_price']} dollari. "
        f"Biglietti non utilizzati: {row['tickets_out']}, biglietti effettivamente usati: {row['ticket_use']}. "
        f"Giorno: {row['day']}, mese: {row['month']}, trimestre: {row['quarter']}."
    )

for _, row in df_ridotto.iterrows():
    raw_text = row_to_text(row)
    embedded_vector = embedding_model.encode(raw_text).tolist()

    # Insert with a custom vector
    collection.data.insert(
        properties={
            "cinema_code": row["cinema_code"],
            "tickets_sold": int(row["tickets_sold"]),
            "tickets_out": int(row["tickets_out"]),
            "show_time": int(row["show_time"]),
            "occu_perc": float(row["occu_perc"]),
            "ticket_price": float(row["ticket_price"]),
            "ticket_use": int(row["ticket_use"]),
            "capacity": int(row["capacity"]),
            "date": row["date"],
            "quarter": int(row["quarter"]),
            "day": int(row["day"]),
            "film_code": row["film_code"],
            "total_sales": float(row["total_sales"]),
            "month": row["month"],
            "raw_text": raw_text
        },
        vector=embedded_vector
    )


In [22]:
### --- 2. USER QUERY --- ###

collection = weaviate_client.collections.get("book_data")

response = collection.query.near_text(
    query = "A book published by DoubleDay books",
    limit = 100
)

for item in response.objects:
    print(item.properties) # Print the properties of each item

In [8]:

collections = weaviate_client.collections.list_all()

print(collections)

{'Book': _CollectionConfigSimple(name='Book', description=None, generative_config=None, properties=[_Property(name='title', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_range_filters=False, index_searchable=True, nested_properties=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_config=_PropertyVectorizerConfig(skip=False, vectorize_property_name=True), vectorizer='text2vec-transformers', vectorizer_configs=None), _Property(name='description', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_range_filters=False, index_searchable=True, nested_properties=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_config=_PropertyVectorizerConfig(skip=False, vectorize_property_name=True), vectorizer='text2vec-transformers', vectorizer_configs=None), _Property(name='authors', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_range_filters=False, index_searchable=True, nested_prope

In [ ]:
### --- 3. GEMINI PROMPT --- ###
#prompt = f"""Domanda: {query}
#Informazioni rilevanti:
#{context}

#Rispondi in modo chiaro:"""

#response = genai.generate_content(model="gemini-1.5-pro", contents=prompt)
#print("Risposta:", response.text)

In [20]:
# Reset schema
import weaviate

weaviate_client = weaviate.connect_to_local()

In [16]:
collection = weaviate_client.collections.get("book_data")
